# Lab 10B: Right for the Right Reasons?

**Before you start: Runtime → Change runtime type → GPU.**

This is the second of three notebooks, and the most important one this week.

The lecture opened with a classifier that tells huskies from wolves with high accuracy, by looking
at the **snow** in the background. High accuracy alone did not reveal the problem. In this notebook
you will meet a model with the same kind of problem, use the tools from the lecture to find out what
it is looking at, fix it, and check the fix. Then we turn the question around: **can we trust the
tools themselves?**

| Part | You will… |
|---|---|
| 1. A model that works perfectly… | train the cat/dog/horse classifier from Lab 2 on a slightly different dataset, and deploy it |
| 2. Detective work | use Grad-CAM, gradient saliency, occlusion and nearest neighbours to find out what the model looks at |
| 3. Test the hypothesis | confirm the explanation with a controlled experiment, and measure which tool found it |
| 4. Fix it, and check the fix | remove the shortcut, and verify with the same tools |
| 5. Can we trust the explanations? | run a *sanity check* on four saliency methods |
| 6. Exam-style questions | check your understanding |

## Setup

In [ ]:
import copy
import math
import random
import time
import zipfile
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# ---- Images: the example images from the lecture, stored in the course repository ----
import os, urllib.request
from PIL import Image

REPO = 'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/'
os.makedirs('images', exist_ok=True)


def get_image(name):
    path = os.path.join('images', name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(REPO + name, path)
    return Image.open(path).convert('RGB')


# ---- The cat/dog/horse dataset from Lab 2 ----
import gdown
if not Path('data').exists():
    gdown.download('https://drive.google.com/uc?id=1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj', 'data.zip', quiet=True)
    with zipfile.ZipFile('data.zip') as z:
        z.extractall('.')
dataset = torchvision.datasets.ImageFolder('data', transform=T.Compose([T.Resize((224, 224)), T.ToTensor()]))
class_names = dataset.classes
X_all = torch.stack([img for img, _ in dataset]).to(device)            # (N, 3, 224, 224), pixels in [0, 1]
y_all = torch.tensor(dataset.targets, device=device)
print('Classes:', class_names, '  images per class:', torch.bincount(y_all).tolist())

# 60% training, 20% validation, 20% test
perm = torch.randperm(len(y_all), generator=torch.Generator().manual_seed(0)).to(device)
n_tr, n_va = int(0.6 * len(perm)), int(0.2 * len(perm))
tr, va, te = perm[:n_tr], perm[n_tr:n_tr + n_va], perm[n_tr + n_va:]
print(f'training {len(tr)}, validation {len(va)}, test {len(te)} images')

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

---
# Part 1 — A model that works perfectly…

Imagine that your group collected the cat, dog and horse photos from three different stock-photo
websites: one for cats, one for dogs and one for horses. Each website adds its own small **logo** to
the corner of every photo. The function `add_logo` below simulates this: website $k$ adds a small
square in its own colour.

In [ ]:
LOGO_COLOURS = torch.tensor([[0.9, 0.1, 0.1], [0.1, 0.8, 0.1], [0.1, 0.2, 0.9]], device=device)   # website 0, 1, 2
LOGO, MARGIN = 28, 6                                   # a 28x28 square, 6 pixels from the bottom-right corner


def add_logo(x, website):
    # x: (N, 3, 224, 224) images in [0, 1]; website: (N,) tensor with the website (0, 1 or 2) of each image
    x = x.clone()
    x[:, :, -LOGO - MARGIN:-MARGIN, -LOGO - MARGIN:-MARGIN] = LOGO_COLOURS[website].view(-1, 3, 1, 1)
    return x


LOGO_MASK = torch.zeros(224, 224, device=device)
LOGO_MASK[-LOGO - MARGIN:-MARGIN, -LOGO - MARGIN:-MARGIN] = 1            # where the logo is (used in Part 3)

idx = tr[:8]
examples = add_logo(X_all[idx], y_all[idx])            # each class comes from its own website
fig, axs = plt.subplots(1, 8, figsize=(18, 2.8))
for ax, x, y in zip(axs, examples, y_all[idx]):
    ax.imshow(x.permute(1, 2, 0).cpu()); ax.set_title(class_names[y]); ax.axis('off')
plt.show()

### Train the classifier

We use exactly the recipe from Lab 2: an ImageNet-pretrained ResNet-18 with a new final layer, Adam
with lr = 1e-3, 10 epochs, random horizontal flips and brightness jitter. The ImageNet normalisation is
built into the model (the first module), so the model takes images with pixels in $[0, 1]$, which makes
the saliency maps easier to interpret.

The training and validation images come from the three websites, so they carry the logos.

In [ ]:
def make_model():
    net = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(512, 3)
    return nn.Sequential(T.Normalize(IMAGENET_MEAN.flatten().tolist(), IMAGENET_STD.flatten().tolist()), net).to(device)


def augment(x):
    flip = torch.rand(len(x), 1, 1, 1, device=device) < 0.5
    x = torch.where(flip, x.flip(-1), x)
    return (x * (0.8 + 0.4 * torch.rand(len(x), 1, 1, 1, device=device))).clamp(0, 1)


def train(logo_fn, epochs=10, seed=0):
    # logo_fn(x, y) adds the logos to a training batch
    set_seed(seed)
    model = make_model()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.1)
    for epoch in range(epochs):
        model.train()
        for b in tr[torch.randperm(len(tr), device=device)].split(32):
            x = logo_fn(augment(X_all[b]), y_all[b])         # the logo is added after the augmentation
            loss = F.cross_entropy(model(x), y_all[b])
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
    return model.eval()


@torch.no_grad()
def accuracy(model, x, y):
    return (torch.cat([model(xb).argmax(1) for xb in x.split(64)]) == y).float().mean().item()


t0 = time.time()
model = train(lambda x, y: add_logo(x, y))               # website = class
print(f'Training time: {time.time() - t0:.0f} s')
print(f'Validation accuracy (photos from the three websites): {accuracy(model, add_logo(X_all[va], y_all[va]), y_all[va]):.1%}')

### Deployment

The validation accuracy looks excellent. Now the model is deployed on **new photos** of cats, dogs and
horses that come from other sources, and therefore have no logo (our test images).

✍️ **Prediction:** what accuracy do you expect on the new photos?

In [ ]:
acc_new = accuracy(model, X_all[te], y_all[te])
print(f'Accuracy on new photos (no logo): {acc_new:.1%}')

### Questions 1

1. Compare the validation accuracy with the accuracy on the new photos. Why didn't the validation set
   warn us?
2. Write down **two** hypotheses that could explain the drop. (Don't look ahead.)

---
# Part 2 — Detective work

We now use four tools to find out what the model looks at. The first three produce a **saliency map**
(a heat map over the image); the fourth looks at the model's **feature space**.

| Tool | Question it answers | Cost |
|---|---|---|
| Grad-CAM (Lab 6, lecture 4.4) | which regions of the last conv layer support the class? | 1 forward + 1 backward pass |
| Gradient saliency (10A) | which pixels is the class score most *sensitive* to? | 1 forward + 1 backward pass |
| Occlusion (lecture 4.2) | how much does the class probability drop if we cover a patch? | one forward pass per patch position |
| Nearest neighbours (lecture 3.1) | which training images does the model consider similar? | one forward pass per image |

Grad-CAM is given (you implemented it in Lab 6). You implement the gradient saliency and the
occlusion map.

In [ ]:
def grad_cam(model, x, class_idx):
    # x: (1, 3, H, W). Grad-CAM on the last convolutional block (layer4) of the ResNet.
    store = {}
    def hook(module, inputs, output):
        store['A'] = output
        output.register_hook(lambda g: store.__setitem__('dA', g))
    h = model[1].layer4.register_forward_hook(hook)
    model.zero_grad()
    model(x)[0, class_idx].backward()
    h.remove()
    w = store['dA'].mean(dim=(2, 3), keepdim=True)                       # channel importance
    cam = F.relu((w * store['A']).sum(dim=1, keepdim=True))              # weighted sum, ReLU
    return F.interpolate(cam, size=x.shape[2:], mode='bilinear')[0, 0].detach()

### 🛠️ TASK 2a: Gradient saliency and occlusion

**Occlusion:** slide a grey square (`patch` × `patch` pixels, value 0.5) over the image with step
`stride`. For each position, record how much the probability of `class_idx` **drops** compared with
the unoccluded image, and add that drop to every pixel the square covered. At the end, divide by how
often each pixel was covered. We process all occluded copies of the image as one batch for speed.

In [ ]:
def gradient_saliency(model, x, class_idx):
    x = x.clone().requires_grad_(True)
    model.zero_grad()
    ???                                                  # backpropagate the logit of class_idx
    return ???                                           # |gradient|, maximum over the colour channels: (H, W)


@torch.no_grad()
def occlusion(model, x, class_idx, patch=32, stride=16):
    H = x.shape[-1]
    p_orig = F.softmax(model(x), dim=1)[0, class_idx]
    positions = [(i, j) for i in range(0, H - patch + 1, stride) for j in range(0, H - patch + 1, stride)]
    occluded = x.repeat(len(positions), 1, 1, 1)
    for n, (i, j) in enumerate(positions):
        ???                                              # grey out the square at (i, j) in copy n
    p_occ = torch.cat([F.softmax(model(b), dim=1)[:, class_idx] for b in occluded.split(64)])
    heat = torch.zeros(H, H, device=x.device)
    count = torch.zeros(H, H, device=x.device)
    for n, (i, j) in enumerate(positions):
        heat[i:i + patch, j:j + patch] += ???            # the drop in probability for this position
        count[i:i + patch, j:j + patch] += 1
    return (heat / count.clamp(min=1)).clamp(min=0)

In [ ]:
# Test for Task 2a
_x = add_logo(X_all[va[:1]], y_all[va[:1]])
_g = gradient_saliency(model, _x, 0)
_o = occlusion(model, _x, y_all[va[0]].item())
assert _g.shape == (224, 224) and (_g >= 0).all()
assert _o.shape == (224, 224) and (_o >= 0).all() and _o.max() > 0, 'the occlusion map is empty: check the drop'
print('✅ Task 2a passed')

### Look at the maps

For six validation images (with logos), we show the image and the three maps for the predicted class.

✍️ **Prediction:** where will the maps be bright?

In [ ]:
def show_maps(model, x, title=''):
    methods = {'Grad-CAM': grad_cam, 'gradient': gradient_saliency, 'occlusion': occlusion}
    fig, axs = plt.subplots(len(x), 4, figsize=(12, 3 * len(x)))
    for r in range(len(x)):
        xi = x[r:r + 1]
        with torch.no_grad():
            c = model(xi).argmax().item()
        axs[r, 0].imshow(xi[0].permute(1, 2, 0).cpu()); axs[r, 0].set_title(f'predicted: {class_names[c]}')
        for k, (name, f) in enumerate(methods.items()):
            m = f(model, xi, c).cpu()
            axs[r, k + 1].imshow(xi[0].permute(1, 2, 0).cpu())
            axs[r, k + 1].imshow(m, cmap='jet', alpha=0.5, vmax=m.quantile(0.99).item() + 1e-12)
            axs[r, k + 1].set_title(name)
    for ax in axs.flat:
        ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()


x_val = add_logo(X_all[va[:6]], y_all[va[:6]])
show_maps(model, x_val, 'Model trained on the website photos, validation images')

### 🛠️ TASK 2b: Nearest neighbours in feature space

A saliency map explains *one* image. Nearest neighbours tell us what the model considers **similar**.
We take the 512-dimensional features before the final layer, and for each query image we find the 5
most similar training images (cosine similarity).

As queries, we use the test images with a **random** logo, so that the logo and the class disagree for
most images. If the model "thinks in animals", the neighbours should have the same class as the query.
If it "thinks in logos", they should come from the same website as the query's logo.

In [ ]:
def features(model, x):
    # The 512-d features before the final layer: replace the final layer by the identity
    backbone = copy.deepcopy(model)
    backbone[1].fc = nn.Identity()
    with torch.no_grad():
        return torch.cat([backbone(b) for b in x.split(64)])


def neighbour_check(model, seed=1):
    g = torch.Generator(device=device).manual_seed(seed)
    logo_q = torch.randint(0, 3, (len(te),), device=device, generator=g)      # a random logo for each query
    f_query = F.normalize(features(model, add_logo(X_all[te], logo_q)), dim=1)
    f_train = F.normalize(features(model, add_logo(X_all[tr], y_all[tr])), dim=1)
    similarity = ???                                     # cosine similarity of every query to every training image: (n_query, n_train)
    nn5 = ???                                            # indices of the 5 most similar training images: (n_query, 5)
    labels = y_all[tr][nn5]                              # the class (= website) of each neighbour
    same_class = (labels == y_all[te][:, None]).float().mean().item()
    same_logo = (labels == logo_q[:, None]).float().mean().item()
    return same_class, same_logo


same_class, same_logo = neighbour_check(model)
print(f'5 nearest neighbours: {same_class:.0%} have the same class as the query, {same_logo:.0%} come from the website of its logo')

### Questions 2

1. Where are the saliency maps bright? Do all three methods agree?
2. What do the nearest neighbours tell you? Which of your hypotheses from Question 1.2 is supported?
3. The occlusion map needed many forward passes per image; Grad-CAM and the gradient one forward and
   one backward pass. Which tools would you use on a whole validation set, and which on a few
   interesting images?

---
# Part 3 — Test the hypothesis, and score the tools

### A controlled experiment

Saliency maps and neighbours suggest an explanation; they don't prove it. A direct test: change
**only the logo** and see what happens to the prediction. We evaluate the test images in three
conditions: with the logo of the correct website, with no logo, and with the logo of a **wrong**
website.

✍️ **Prediction:** the accuracy in the third condition?

In [ ]:
def three_conditions(model, seed=2):
    g = torch.Generator(device=device).manual_seed(seed)
    wrong = (y_all[te] + torch.randint(1, 3, (len(te),), device=device, generator=g)) % 3
    return {'correct logo': accuracy(model, add_logo(X_all[te], y_all[te]), y_all[te]),
            'no logo': accuracy(model, X_all[te], y_all[te]),
            'wrong logo': accuracy(model, add_logo(X_all[te], wrong), y_all[te])}


results = {'trained on website photos': three_conditions(model)}
for name, r in results.items():
    print(f'{name:28s}', {k: f'{v:.0%}' for k, v in r.items()})

### Which tool found the shortcut?

Here we know the truth: the logo occupies a known 28×28 square, only 1.6% of the image. So we can
**score** each method: what fraction of the map's total "mass" lies inside the logo? A method that
ignores the logo would give about 1.6%; a method that finds it should give much more.

### 🛠️ TASK 3a: Score the saliency maps

In [ ]:
def fraction_in_logo(saliency_map):
    return ???                                           # sum of the map inside LOGO_MASK, divided by the sum of the whole map


def score_methods(model, n=30, seed=3):
    g = torch.Generator(device=device).manual_seed(seed)
    logo_q = torch.randint(0, 3, (n,), device=device, generator=g)
    x = add_logo(X_all[te[:n]], logo_q)
    scores = {}
    for name, f in {'Grad-CAM': grad_cam, 'gradient': gradient_saliency, 'occlusion': occlusion}.items():
        vals = []
        for i in range(n):
            with torch.no_grad():
                c = model(x[i:i + 1]).argmax().item()
            vals.append(fraction_in_logo(f(model, x[i:i + 1], c)))
        scores[name] = float(np.mean(vals))
    return scores


print(f'Logo area: {LOGO_MASK.mean().item():.1%} of the image')
tool_scores = {'trained on website photos': score_methods(model)}
print({k: f'{v:.1%}' for k, v in tool_scores['trained on website photos'].items()})

In [ ]:
# Test for Task 3a
assert abs(fraction_in_logo(LOGO_MASK) - 1.0) < 1e-6 and abs(fraction_in_logo(torch.ones(224, 224, device=device)) - LOGO_MASK.mean().item()) < 1e-6
print('✅ Task 3a passed')

### Questions 3

1. What happened with the **wrong** logo? What does this experiment show that the saliency maps could
   not?
2. Compare each method's score with the logo's area (1.6%). Which method points at the logo most clearly?
   Explain the difference. (Hint: what question does each method ask? The gradient: what happens if **one
   pixel** changes **a little**? Occlusion: what happens if a **whole region** is removed?)
3. Why is it useful to test explanation methods on a problem where you **know** the answer?

---
# Part 4 — Fix it, and check the fix

In a real project you could crop the logos away, collect photos from more websites, or make sure every
website contributes every class. Here we use a quick fix that breaks the correlation: during training,
every photo gets the logo of a **random** website. The logo is still there, but it no longer tells the
model anything about the class.

✍️ **Prediction:** what will the three accuracies be for the fixed model? And what will the scores of the
three tools be?

In [ ]:
t0 = time.time()
model_fixed = train(lambda x, y: add_logo(x, torch.randint(0, 3, y.shape, device=device)))
print(f'Training time: {time.time() - t0:.0f} s')

results['trained with random logos'] = three_conditions(model_fixed)
tool_scores['trained with random logos'] = score_methods(model_fixed)
same_class_f, same_logo_f = neighbour_check(model_fixed)

print(f'{"":28s}{"correct logo":>14s}{"no logo":>10s}{"wrong logo":>12s}')
for name, r in results.items():
    print(f'{name:28s}{r["correct logo"]:14.0%}{r["no logo"]:10.0%}{r["wrong logo"]:12.0%}')
print(f'\nFraction of saliency inside the logo (logo area {LOGO_MASK.mean().item():.1%}):')
for name, s in tool_scores.items():
    print(f'{name:28s}', {k: f'{v:.1%}' for k, v in s.items()})
print(f'\n5-NN, fixed model: {same_class_f:.0%} same class, {same_logo_f:.0%} same website as the logo')
show_maps(model_fixed, x_val, 'Model trained with random logos, the same validation images')

### Questions 4

1. Did the fix work? Use the three accuracies, the tool scores and the neighbours.
2. The fixed model's validation accuracy (with the correct logos) may be *lower* than the original model's.
   Is the fixed model worse?
3. Our fix assumed we knew what the shortcut was. In a real project you don't. Which of the tools would you
   use to search for unknown shortcuts, and which kind of controlled experiment would you then run?
4. **Project perspective:** name one possible shortcut in **your** project's data (think about how and where
   the images were collected: scanner, hospital, camera, time of day, background, image size, text or
   markers in the image).

---
# Part 5 — Can we trust the explanations?

Part 3 showed that methods can *miss* something. They can also mislead in the opposite way: produce a
convincing picture that has little to do with what the model learned. Adebayo et al. (2018, *Sanity
Checks for Saliency Maps*) proposed a simple test:

> If an explanation describes what the **model** learned, it must change when we destroy what the model
> learned. So: randomise the weights of the network, layer by layer from the top, and check whether the
> saliency map changes.

We test four methods on an ImageNet-pretrained ResNet-18 and the cat from 10A:

* **Grad-CAM** and **gradient saliency** (as above);
* **gradient × input**: the gradient multiplied by the pixel values (a popular variant);
* **guided backpropagation**: the pixel-level part of *Guided Grad-CAM* (lecture 4.4). During the
  backward pass, every ReLU lets only **positive** gradients through (an ordinary ReLU passes the
  gradient wherever its *input* was positive). This produces much sharper, cleaner pictures than the
  plain gradient. We implement it with a backward hook on every ReLU.

We compare each map with the map of the trained network using the **rank correlation** (1 = the same
ordering of pixels, 0 = unrelated).

✍️ **Prediction:** which methods will still give a similar map when the top layers are random?

In [ ]:
from scipy.stats import spearmanr

imagenet_model = nn.Sequential(T.Normalize(IMAGENET_MEAN.flatten().tolist(), IMAGENET_STD.flatten().tolist()),
                               torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)).to(device).eval()
for m in imagenet_model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False
cat_img = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor()])(get_image('Cat03.jpg')).unsqueeze(0).to(device)
with torch.no_grad():
    cat_class = imagenet_model(cat_img).argmax().item()
print('ResNet-18 predicts class', cat_class, torchvision.models.ResNet18_Weights.IMAGENET1K_V1.meta['categories'][cat_class])


def gradient_x_input(model, x, class_idx):
    x = x.clone().requires_grad_(True)
    model.zero_grad(); model(x)[0, class_idx].backward()
    return (x.grad * x).abs().amax(dim=1)[0].detach()


def guided_backprop(model, x, class_idx):
    hooks = [m.register_full_backward_hook(lambda mod, g_in, g_out: (g_in[0].clamp(min=0),))
             for m in model.modules() if isinstance(m, nn.ReLU)]          # only positive gradients pass the ReLUs
    sal = gradient_saliency(model, x, class_idx)
    for h in hooks:
        h.remove()
    return sal


METHODS = {'gradient': gradient_saliency, 'gradient × input': gradient_x_input,
           'guided backprop': guided_backprop, 'Grad-CAM': grad_cam}

### 🛠️ TASK 5a: Randomise the network, layer by layer

Write `randomise(model, block_names)`: return a **copy** of the model in which every layer inside the named
blocks of the ResNet (`model[1].fc`, `model[1].layer4`, …) gets new random weights. Every PyTorch layer
with weights has a method `reset_parameters()` that does exactly that.

In [ ]:
def randomise(model, block_names, seed=1):
    torch.manual_seed(seed)
    m = copy.deepcopy(model)
    for name in block_names:
        for layer in getattr(m[1], name).modules():
            if ???:                                       # only layers that have weights to reset
                ???
    return m

In [ ]:
# Test for Task 5a
_m = randomise(imagenet_model, ['fc'])
assert not torch.equal(_m[1].fc.weight, imagenet_model[1].fc.weight), 'the fc layer was not randomised'
assert torch.equal(_m[1].layer4[0].conv1.weight, imagenet_model[1].layer4[0].conv1.weight), 'layer4 should be unchanged'
print('✅ Task 5a passed')

stages = [('trained', []), ('fc', ['fc']), ('+ layer4', ['fc', 'layer4']), ('+ layer3', ['fc', 'layer4', 'layer3']),
          ('+ layer2', ['fc', 'layer4', 'layer3', 'layer2']), ('all', ['fc', 'layer4', 'layer3', 'layer2', 'layer1', 'conv1', 'bn1'])]
reference = {k: f(imagenet_model, cat_img, cat_class).cpu() for k, f in METHODS.items()}
corr = {k: [] for k in METHODS}
fig, axs = plt.subplots(len(METHODS), len(stages), figsize=(3 * len(stages), 3 * len(METHODS)))
for j, (stage, blocks) in enumerate(stages):
    m = randomise(imagenet_model, blocks)
    for i, (k, f) in enumerate(METHODS.items()):
        sal = f(m, cat_img, cat_class).cpu()
        corr[k].append(spearmanr(sal.flatten().numpy(), reference[k].flatten().numpy())[0])
        axs[i, j].imshow(sal, cmap='gray', vmax=sal.quantile(0.99).item() + 1e-12); axs[i, j].axis('off')
        axs[i, j].set_title(f'{k}\nrandomised: {stage}', fontsize=9)
plt.tight_layout(); plt.show()

print(f'{"rank correlation with the trained map":40s}' + ''.join(f'{s:>10s}' for s, _ in stages))
for k, v in corr.items():
    print(f'{k:40s}' + ''.join(f'{c:10.2f}' for c in np.nan_to_num(v)))

### Questions 5

1. Which method's map hardly changed when the top layers (fc, layer4) were randomised? What does that mean
   for what it shows?
2. Which methods passed the sanity check?
3. Guided backprop produced the cleanest, most "cat-like" picture of all methods. Why is that **not** evidence
   that it is the best explanation?
4. Summarise Parts 3 and 5 in one sentence of advice for your project report.

---
# Part 6 — Exam-style questions

**Q1.** A classifier reaches 99% validation accuracy, but Grad-CAM shows that it mostly looks at a watermark
that only appears on the images of one class. Why did the validation accuracy not reveal this?

* (a) Because Grad-CAM is wrong.
* (b) Because the validation images contain the same watermark–class correlation as the training images.
* (c) Because the validation set is too large.
* (d) Because the model overfits.

**Q2.** How does an occlusion saliency map decide that a region is important?

* (a) By the size of the gradient of the class score at the pixels of that region.
* (b) By how much the probability of the class drops when the region is covered.
* (c) By how strongly the first-layer filters respond in that region.
* (d) By the distance to the nearest training image.

**Q3.** A saliency method produces almost the same map for a trained network and for the same network with a
randomly initialised final layer. What can you conclude?

* (a) The method is very robust and therefore trustworthy.
* (b) The map does not depend on what the final layer has learned, so it cannot explain the class decision.
* (c) The final layer is not used by the network.
* (d) The network is overfitting.

**Q4.** In feature space (the layer before the final layer), the nearest neighbours of a dog image with a "cat
website" logo are cat images. What does this indicate?

* (a) The feature space is organised by animal, as it should be.
* (b) The model represents images mainly by the logo.
* (c) The dog image is mislabelled.
* (d) Cosine similarity is the wrong distance measure.

---
**Next:** the optional notebook **10C** (texture and style transfer), and this week's pages in your
**iteration log**, where you use these tools on your own model.